# Lab 6 - Task 2: Gaussian Filter

**Theory:**  
A Gaussian filter smooths an image by averaging pixel values, with more weight given to pixels closer to the center of the kernel. It is used to reduce noise and blur.

**Objectives:**
1. Apply a Gaussian filter using `cv2.GaussianBlur()`.
2. Experiment with different kernel sizes (3x3, 5x5) and sigma values.
3. Compare results to the box filter and describe the differences.
4. Display all results side by side.

In [ ]:
import cv2
import numpy as np
import copy
import matplotlib.pyplot as plt

## Helper Functions

In [ ]:
def box_filt(n):
    if n <= 0 or n % 2 == 0:
        raise ValueError("Window size must be a positive odd integer.")
    return np.ones((n, n), np.float32) / (n * n)


def gauss_filt(n, sigma=1.0):
    """
    Creates a Gaussian filter kernel of size n x n with standard deviation sigma.
    """
    if n <= 0 or n % 2 == 0:
        raise ValueError("Window size must be a positive odd integer.")
    ax = np.linspace(-(n - 1) / 2., (n - 1) / 2., n)
    xx, yy = np.meshgrid(ax, ax)
    kernel = np.exp(-0.5 * (np.square(xx) + np.square(yy)) / np.square(sigma))
    kernel = kernel / np.sum(kernel)
    return kernel


def apply_filters(image_input, kernel, filt_size):
    pad_size = int(np.ceil(filt_size / 2))
    image_padded = np.pad(
        image_input,
        pad_width=((pad_size, pad_size), (pad_size, pad_size)),
        mode='symmetric'
    )
    image_out = copy.deepcopy(image_input)
    row, column = image_input.shape
    for i in range(row):
        for j in range(column):
            patch = image_padded[i:i + filt_size, j:j + filt_size]
            image_out[i, j] = np.sum(kernel * patch)
    return image_out

## Load Image

In [ ]:
image = cv2.imread('image.png', cv2.IMREAD_GRAYSCALE)

if image is None:
    image = np.random.randint(50, 200, (256, 256), dtype=np.uint8)
    print("No image file found — using a synthetic test image.")

## Part 2A: cv2.GaussianBlur — Different Kernel Sizes

In [ ]:
gauss_3x3 = cv2.GaussianBlur(image, (3, 3), 0)
gauss_5x5 = cv2.GaussianBlur(image, (5, 5), 0)
gauss_9x9 = cv2.GaussianBlur(image, (9, 9), 0)

fig, axes = plt.subplots(1, 4, figsize=(16, 4))
for ax, img, title in zip(
    axes,
    [image, gauss_3x3, gauss_5x5, gauss_9x9],
    ['Original', 'Gaussian 3x3', 'Gaussian 5x5', 'Gaussian 9x9']
):
    ax.imshow(img, cmap='gray')
    ax.set_title(title)
    ax.axis('off')
plt.tight_layout()
plt.show()

## Part 2B: cv2.GaussianBlur — Different Sigma Values (5x5 kernel)

In [ ]:
sigma_values = [0.5, 1, 2, 5]
fig, axes = plt.subplots(1, len(sigma_values) + 1, figsize=(18, 4))

axes[0].imshow(image, cmap='gray')
axes[0].set_title('Original')
axes[0].axis('off')

for ax, sigma in zip(axes[1:], sigma_values):
    blurred = cv2.GaussianBlur(image, (5, 5), sigma)
    ax.imshow(blurred, cmap='gray')
    ax.set_title(f'Gaussian 5x5\nσ={sigma}')
    ax.axis('off')

plt.tight_layout()
plt.show()

## Part 2C: Custom Gaussian Kernel vs Box Filter (same size)

In [ ]:
filt_size = 5
image_float = image.astype(np.float32) / 255.0

box_kernel = box_filt(filt_size)
gauss_kernel = gauss_filt(filt_size, sigma=1.0)

print(f"Box kernel ({filt_size}x{filt_size}):")
print(np.round(box_kernel, 4))
print(f"\nGaussian kernel ({filt_size}x{filt_size}, σ=1.0):")
print(np.round(gauss_kernel, 4))

In [ ]:
# Apply with cv2.filter2D for speed (functionally equivalent to apply_filters)
box_result = cv2.filter2D(image, -1, box_kernel)
gauss_result = cv2.filter2D(image, -1, gauss_kernel)

fig, axes = plt.subplots(1, 3, figsize=(14, 4))
for ax, img, title in zip(
    axes,
    [image, box_result, gauss_result],
    ['Original', f'Box Filter {filt_size}x{filt_size}', f'Gaussian Filter {filt_size}x{filt_size} σ=1']
):
    ax.imshow(img, cmap='gray')
    ax.set_title(title)
    ax.axis('off')
plt.tight_layout()
plt.show()

## Discussion: Box Filter vs Gaussian Filter

| Property | Box Filter | Gaussian Filter |
|---|---|---|
| Weights | Uniform (all equal) | Bell-shaped (center has more weight) |
| Blur quality | Can produce block artifacts | Smoother, more natural blur |
| Edge preservation | Poor | Slightly better |
| Control | Kernel size only | Kernel size + sigma |

**Key takeaway:** The Gaussian filter produces a smoother and more visually natural blur because it gives higher importance to pixels closer to the center. The box filter treats all neighbors equally, which can introduce ringing or block-like artifacts. Increasing sigma in the Gaussian filter widens the bell curve, resulting in stronger blurring.